## Requirements

- **Python ≥ 3.11**
- **Packages:** `pip install requests rasterio numpy matplotlib pystac-client pandas`
- **Network:** HTTP access to `kanopia.org` (STAC API) and `lab.kanopia.org` (COG files)
- **Credentials:** Basic Auth required for COG files — `COG_USER` / `COG_PASS` set in the config cell

# STAC Test — Panama 2024–2025 — Python

End-to-end connectivity test for the Kanopia STAC API:

1. Ping the STAC root — confirm the API is reachable
2. Search for all items over **Panama** between **2024-01-01 and 2025-12-31**
3. List all **RGB COG** assets found
4. Render a low-resolution **preview** of the first RGB COG

In [ ]:
# ── Install packages (safe to re-run) ────────────────────────────────────────
import sys, subprocess
pkgs = ["requests", "rasterio", "numpy", "matplotlib", "pystac-client", "pandas"]
subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "-q"])
print("Packages ready.")

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import rasterio
from rasterio.enums import Resampling
from pystac_client import Client
from urllib.parse import urlparse, urlunparse

# GDAL network settings for remote COG reads
os.environ.setdefault("GDAL_HTTP_TIMEOUT",            "120")
os.environ.setdefault("GDAL_HTTP_MAX_RETRY",          "3")
os.environ.setdefault("GDAL_HTTP_RETRY_DELAY",        "5")
os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")

def add_auth(url, user, pwd):
    """Embed Basic Auth credentials into a URL."""
    p = urlparse(url)
    if p.username:
        return url
    netloc = f"{user}:{pwd}@{p.netloc}"
    return urlunparse((p.scheme, netloc, p.path, p.params, p.query, p.fragment))

print("Imports OK.")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
STAC_API_URL = "https://kanopia.org/stac-fastapi-pgstac/api/v1/pgstac/"

# Workshop credentials
COG_USER = "panama"
COG_PASS = "panama123"

# Spatial extent: Panama [west, south, east, north]
BBOX = [-83.05, 7.20, -77.15, 9.65]

# Temporal extent
DATETIME = "2024-01-01T00:00:00Z/2025-12-31T23:59:59Z"

print(f"STAC endpoint : {STAC_API_URL}")
print(f"Bounding box  : {BBOX}  (Panama)")
print(f"Date range    : {DATETIME}")

---
## Step 1 — Ping the STAC API

A simple GET to the root URL confirms the API is reachable and returns its title and version.

In [ ]:
# ── Step 1: Ping STAC root ────────────────────────────────────────────────────
resp = requests.get(STAC_API_URL, timeout=15)
resp.raise_for_status()
catalog = resp.json()

print(f"  Title   : {catalog.get('title', '?')}")
print(f"  Version : {catalog.get('stac_version', '?')}")
print(f"  ID      : {catalog.get('id', '?')}")
print("\nSTAC API is reachable.")

---
## Step 2 — Search over Panama (2024–2025)

Search the STAC API with a **bounding box** covering Panama and a **date range** of 2024–2025.
No collection filter is applied — all matching items across every project are returned.

In [ ]:
# ── Step 2: Search Panama + 2024-2025 ─────────────────────────────────────────
client = Client.open(STAC_API_URL)

search = client.search(bbox=BBOX, datetime=DATETIME, max_items=500)

try:
    items = list(search.item_collection())
except AttributeError:
    items = list(search.get_all_items())  # fallback for older pystac-client

print(f"Found {len(items)} item(s) over Panama (2024\u20132025).")

---
## Step 3 — List RGB COG assets

Filter assets to **optimised RGB COGs** (`.cog.tif`, excluding `raw` and `lowres` variants).

In [ ]:
# ── Step 3: List RGB COG assets ───────────────────────────────────────────────
RGB_PAT     = re.compile(r"rgb.*\.cog\.tif",   re.I)
EXCLUDE_PAT = re.compile(r"raw|lowres|preview", re.I)

rgb_assets = []
for item in items:
    coll  = getattr(item, "collection_id", None) or item.get("collection")
    props = getattr(item, "properties", {}) or item.get("properties", {})
    dt    = getattr(item, "datetime", None) or props.get("datetime") or "?"
    dt_s  = str(dt)[:10]

    item_assets = item.assets if hasattr(item, "assets") else item.get("assets", {})
    if not isinstance(item_assets, dict):
        continue
    for key, asset in item_assets.items():
        href = asset.href if hasattr(asset, "href") else asset.get("href", "")
        if RGB_PAT.search(href) and not EXCLUDE_PAT.search(href) and "/share/1" in href:
            rgb_assets.append(dict(collection=coll, date=dt_s, href=href))

print(f"Found {len(rgb_assets)} RGB COG asset(s):\n")
if rgb_assets:
    df = pd.DataFrame(rgb_assets)
    pd.set_option("display.max_colwidth", 120)
    print(df[["collection", "date", "href"]].to_string(index=False))

---
## Step 4 — COG preview

Open the first RGB COG via `/vsicurl/` (no local download), read at reduced resolution
using GDAL overviews, and display as an RGB image.

In [ ]:
# ── Step 4: COG preview of first RGB COG ──────────────────────────────────────
if not rgb_assets:
    print("No RGB COG assets found \u2014 adjust BBOX or DATETIME.")
else:
    first    = rgb_assets[0]
    url_auth = add_auth(first["href"], COG_USER, COG_PASS)

    print(f"Collection : {first['collection']}")
    print(f"Date       : {first['date']}")
    print(f"File       : {first['href'].split('/')[-1]}")

    with rasterio.open(f"/vsicurl/{url_auth}") as src:
        print(f"\nDimensions : {src.width:,} x {src.height:,} px")
        print(f"CRS        : {src.crs}")
        print(f"Bands      : {src.count}")

        # Read at ~512 px wide — GDAL selects the appropriate overview level
        out_w = 512
        out_h = max(1, int(src.height * out_w / src.width))
        data  = src.read([1, 2, 3], out_shape=(3, out_h, out_w),
                         resampling=Resampling.nearest)

    def pct_stretch(band, lo=2, hi=98):
        vmin, vmax = np.nanpercentile(band, [lo, hi])
        return np.clip((band.astype(float) - vmin) / (vmax - vmin + 1e-6), 0, 1)

    rgb = np.dstack([pct_stretch(data[i]) for i in range(3)])

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(rgb)
    ax.set_title(
        f"{first['collection']}  |  {first['date']}\n{first['href'].split('/')[-1]}",
        fontsize=11
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()